In [0]:
# Ordena lo subido y simula la llegada de lotes al landing
CAT  = "electrocasa_dev"
BASE = f"/Volumes/{CAT}/default/vol_landing"
STG  = f"/Volumes/{CAT}/default/vol_staging"   # lotes pendientes (fuera del landing)
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CAT}.default.vol_staging COMMENT 'Lotes pendientes de llegar al landing'")

# 1) Mover TODO lo que quedó por error dentro de vol_landing/ventas/ a staging
for f in dbutils.fs.ls(f"{BASE}/ventas"):
    dbutils.fs.mv(f.path, f"{STG}/{f.name}", recurse=True)

display(dbutils.fs.ls(STG))

In [0]:
def cargar_lote(n):
    """Copia al landing solo los archivos del lote n (y el snapshot que corresponde)."""
    for fuente in ["ventas", "empleados", "devoluciones", "resenas"]:
        for f in dbutils.fs.ls(f"{STG}/{fuente}"):
            if f"lote_0{n}" in f.name:
                dbutils.fs.cp(f.path, f"{BASE}/{fuente}/{f.name}")
    snapshot = {1: "2026-03-01", 2: "2026-08-01"}.get(n)
    if snapshot:
        nombre = f"catalogo_productos_{snapshot}.json"
        dbutils.fs.cp(f"{STG}/catalogo/{nombre}", f"{BASE}/catalogo/{nombre}")
    if n == 1:
        dbutils.fs.cp(f"{STG}/sucursales.csv", f"{BASE}/sucursales/sucursales.csv")
    # Ver qué quedó en el landing
    for c in ["ventas", "empleados", "devoluciones", "resenas", "catalogo", "sucursales"]:
        print(c, [x.name for x in dbutils.fs.ls(f"{BASE}/{c}")])

cargar_lote(1)

In [0]:
cargar_lote(2)

In [0]:
cargar_lote(3)